# 06 - Generation And API (Hue Foods RAG MVP)

Notebook này chạy **full runtime path thật** của Phase 6 qua API: câu hỏi -> retrieval thật (Qdrant + E5) -> ContextBuilder thật -> OpenAI generator thật (`gpt-5.4-nano` qua OpenAI Agents SDK) -> JSON sources.

**Prerequisite**

- Khởi động Jupyter từ terminal đã export `OPENAI_API_KEY` vào environment (notebook không đọc `.env`, không dùng `load_dotenv`). Nếu thiếu key, notebook fail actionable ngay - không có fallback.
- Qdrant local đang chạy (collection `hue_foods_e5_small_384`, 572 points) và E5 đã cache.

**Chi phí**

- Mỗi Run All gọi **đúng 1** OpenAI call (`POST /api/chat`). Chi phí ước tính dưới 0,001 USD mỗi call (dựa trên smoke 2026-08-13: ~1.100-1.400 input tokens, ~100-500 output tokens).

**Kết quả mong đợi khi Run All**

- `/health` trả `ok` với các component ready.
- `/api/chat` trả HTTP 200: answer tiếng Việt grounded, `sources` theo context order, `retrieval_debug` với profile/model thật.
- Nếu có lỗi provider, response trả safe error shape; notebook không retry.

Lần chạy đầu tiên có thể mất vài chục giây vì E5 load từ cache vào memory.


In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Khong tim thay thu muc backend/. Hay mo notebook nay tu repo root "
        "hoac tu thu muc notebooks/."
    )
print(f"backend on path: {sys.path[0]}")


## Kiểm tra key và offline mode

Cell dưới chỉ kiểm tra **presence** của `OPENAI_API_KEY` (không in giá trị) và đặt `HF_HUB_OFFLINE=1` để E5/MiniLM chỉ dùng local cache. Nếu key thiếu, notebook dừng với thông báo rõ ràng.


In [ ]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"

has_key = bool(os.environ.get("OPENAI_API_KEY", "").strip())
print("OPENAI_API_KEY present:", has_key)
if not has_key:
    raise RuntimeError(
        "Thieu OPENAI_API_KEY trong environment. Hay khoi dong Jupyter tu "
        "terminal da export key (export OPENAI_API_KEY=...) roi chay lai."
    )


## Câu hỏi

Người dùng chỉ cần sửa đúng một biến dưới đây rồi Run All. Không cần tự tạo chunk_id, evidence JSON hay available_source_ids - API tự chạy retrieval và build context.


In [ ]:
question = "Bún bò Huế có gì đặc biệt?"
print("question:", question)


## Gọi `/api/chat` qua app thật

Cell dưới dùng FastAPI `TestClient` với app thật và lifespan thật: lifespan build retrieval stack một lần (Qdrant + E5 thật), rồi `/health` đọc cached readiness và `POST /api/chat` chạy đúng 1 retrieval + 1 OpenAI call. In ra answer, sources projection, session_id và retrieval_debug - không in prompt, raw SDK response, header hay token.


In [ ]:
import time

from fastapi.testclient import TestClient

from api.app import app

with TestClient(app) as client:
    health = client.get("/health")
    print("health status:", health.status_code)
    print("health body:", health.json())

    started = time.monotonic()
    response = client.post("/api/chat", json={"query": question})
    elapsed = round(time.monotonic() - started, 1)
    print("chat status:", response.status_code)
    print("elapsed seconds:", elapsed)

    body = response.json()
    if response.status_code == 200:
        print("answer:", body["answer"])
        print("sources:")
        for source in body["sources"]:
            print("  -", source["title"], "|", source["section"],
                  "| score:", source["score"], "|", source["chunk_id"])
        print("session_id:", body["session_id"])
        print("retrieval_debug:", body["retrieval_debug"])
    else:
        print("error body (safe shape):", body)


## Checklist xác nhận Phase 6

1. `OPENAI_API_KEY present: True` được in ra (không bao giờ in giá trị key).
2. `/health` trả `ok` với `qdrant: ready`, `retrieval: ready`, `generator: configured`.
3. `/api/chat` trả HTTP 200 với answer tiếng Việt, `sources` hợp lệ theo context order và `retrieval_debug` đúng profile/model thật.
4. Nếu provider trả structured output không hợp lệ, response là HTTP 502 với safe error shape - notebook không retry.
5. Đúng 1 OpenAI call cho mỗi Run All.
